# Basic RAG — Naive Retrieval-Augmented Generation (1 of 6)

## RAG Workshop Series

This notebook is part of a 6-notebook series (split from the original `RAG.ipynb`), each one runnable on its own in Google Colab:

1. **`basic_RAG.ipynb`** — naive vector RAG: chunk → embed → cosine similarity → prompt → LLM
2. **`hybrid_search_RAG.ipynb`** — BM25 keyword search + semantic search fusion + cross-encoder reranking
3. **`query_and_chunking_RAG.ipynb`** — query rewriting, advanced chunking strategies, metadata filtering
4. **`agentic_RAG.ipynb`** — Corrective RAG (CRAG), Adaptive RAG (routing), Agentic RAG (ReAct loop)
5. **`pdf_chroma_RAG.ipynb`** — build RAG over a real PDF, store vectors persistently in Chroma
6. **`rag_when_to_use.ipynb`** — reference: when RAG is (and isn't) the right tool

Each notebook installs its own dependencies and rebuilds whatever context it needs, so you can open any one directly without running the others first.

#Happy Learning!

## What Are We Building?

Imagine that we have a small knowledge base about AWS services.

A user asks:

> "Which AWS service should I use to decouple applications?"

Our system will:

1. Break documents into chunks.
2. Convert the chunks into embeddings.
3. Convert the question into an embedding.
4. Compare the question with every chunk.
5. Retrieve the most relevant chunks.

That is the **retrieval** part of Retrieval-Augmented Generation.

The architecture looks like this:

📄 Documents → ✂️ Chunks → 🔢 Embeddings → 🔍 Vector Search → 📚 Relevant Context → 📝 Prompt → 🤖 LLM → 💬 Answer

In a production application, you might use Pinecone, OpenSearch, pgvector, Qdrant, or another vector database.

For learning, we don't need any of them.

We will store our embeddings in memory and use cosine similarity.

## Step 1: Open Google Colab

Create a new Google Colab notebook.

Run:

In [ ]:
!pip install -q sentence-transformers openai

We will use the `sentence-transformers` library to generate embeddings.

No API key is required.

## Step 2: Create Our Knowledge Base

Let's create a tiny collection of documents.

In [ ]:
documents = [
    """
    Amazon S3 is an object storage service designed for storing
    and retrieving files. It provides high durability and is
    commonly used for backups, static websites, data lakes,
    and application assets.
    """,

    """
    Amazon SQS is a managed message queue service.

    It allows applications to communicate asynchronously.

    Producers send messages to a queue and consumers process
    those messages independently.

    SQS is commonly used to decouple distributed applications.
    """,

    """
    AWS Lambda is a serverless compute service.

    Developers upload code and AWS executes the code in response
    to events.

    Lambda automatically manages servers and scales applications
    based on incoming requests.
    """,

    """
    Amazon DynamoDB is a managed NoSQL database.

    It provides low-latency access to data and automatically
    scales to handle large workloads.
    """
]

Obviously, real RAG systems contain thousands or millions of documents.

But the mechanism is exactly the same.

## Step 3: Chunk the Documents

Why do we need chunks?

Because embedding an entire book or a 200-page PDF as one vector would produce a poor representation for individual questions.

Instead, RAG systems divide documents into smaller pieces.

Let's write a very simple chunker.

In [ ]:
documents[0]

In [ ]:
def chunk_text(text, chunk_size=250):
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


chunks = []

for document in documents:
    chunks.extend(chunk_text(document))


print("Number of chunks:", len(chunks))

for index, chunk in enumerate(chunks):
    print(f"\nCHUNK {index}")
    print(chunk)

Production systems usually use more sophisticated strategies.

For example:

- token-based chunking
- overlapping chunks
- recursive text splitting
- semantic chunking

But our goal is to understand the mechanism first.

## Step 4: Generate Embeddings

Now we need to convert text into vectors.

We will use a small embedding model.

### Core Specifications

Dimensions: 384 output vector size

Parameters: Around 22 million

Max Tokens: 256 token sequence length.

File Size: Roughly 70 MB

Speed: Fast inference on CPU and edge devices

### Common Uses Semantic Search:

Find text with similar meanings instead of exact keyword matches.

Clustering: Group similar user reviews, support tickets, or articles together.

RAG Systems: Build local or lightweight retrieval-augmented generation pipelines.


In [ ]:
%%capture
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Now generate embeddings for our chunks.

In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

You should see something similar to: (4, 384)


What does that mean?

We have four chunks.

Each chunk has been converted into a vector containing 384 numbers.

Something like this: [0.023, -0.041, 0.087, ...]

Humans cannot interpret these numbers directly.

But mathematically, texts with similar meanings tend to have vectors that are closer together.

That is what makes semantic search possible.

## Step 5: Ask a Question

Let's ask:

In [ ]:
question = "Which AWS service can help decouple applications?"

Convert the question into an embedding.

In [ ]:
question_embedding = embedding_model.encode([question])

Now we have:

Documents → Vectors

Question → Vector

We need to find which document vectors are closest to the question vector.

## Step 6: Perform Similarity Search

We will use cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(
    question_embedding,
    chunk_embeddings
)[0]

Let's inspect the results.

In [ ]:
for index, score in enumerate(scores):
    print(round(score, 3), chunks[index][:200])

You should see the SQS document receive the highest similarity score.

Now retrieve the best chunk.

In [ ]:
best_chunk_index = scores.argmax()

retrieved_chunk = chunks[best_chunk_index]

print(retrieved_chunk)

Congratulations.

You just built semantic retrieval.

This is the foundation of a RAG system.

## Let's Make It Reusable

Let's wrap everything into a function.

In [ ]:
def search(question, top_k=2):

    question_embedding = embedding_model.encode([question])

    scores = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    top_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index, score in zip(top_indices, scores[top_indices]):
        results.append({
            "score": float(score),
            "text": chunks[index]
        })

    return results

Now try asking different questions.

In [ ]:
questions = [
    "Where should I store application files?",
    "How can I run code without managing servers?",
    "Which database provides low latency access?",
    "How can microservices communicate asynchronously?"
]

for question in questions:

    print("\nQUESTION:", question)

    results = search(question)

    for result in results:
        print(
            round(result["score"], 3),
            result["text"][:150]
        )

You now have a tiny semantic search engine.
Yaaayee!

In [ ]:
def build_prompt(question, retrieved_chunks):

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    Answer the question using only the context below.

    CONTEXT:

    {context}

    QUESTION:

    {question}
    """

    return prompt

Let's generate the prompt.

In [ ]:
question = "Which AWS service should I use to decouple applications?"

results = search(question)

retrieved_chunks = [
    result["text"]
    for result in results
]

prompt = build_prompt(
    question,
    retrieved_chunks
)

print(prompt)

This prompt could now be sent to an LLM.


We will use openAI library to call llama 3.1-8B-instruct an 8B parameter model which is suitable for simple chatbot usecases. Although the library from openAI the service we use is nscale.

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get('NSCALE_API'),
    base_url="https://inference.api.nscale.com/v1",
)

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_tokens=500,
    temperature=0.7,
)

print(response.choices[0].message.content)


📄 Documents → ✂️ Chunks → 🔢 Embeddings → 🔍 Vector Search → 📚 Relevant Context → 📝 Prompt → 🤖 LLM → 💬 Answer

## Five Experiments You Should Try

Don't stop after running the notebook.

Change it.

First, add documents that use similar terminology.

For example:

SQS decouples applications.

EventBridge connects applications using events.

SNS distributes messages to multiple subscribers.

Does the correct document still rank first?

Second, change the chunk size.

Try:

In [ ]:
chunk_size = 50

Then:

In [ ]:
chunk_size = 500

What happens to retrieval quality?

Third, retrieve more documents.

Change:

In [ ]:
top_k = 1

to:

In [ ]:
top_k = 3

Would sending more context always produce a better answer?

Fourth, add irrelevant documents.

Does retrieval quality change?

Fifth, ask ambiguous questions.

For example:

Which service should I use for messaging?

Now you are starting to encounter the problems that real RAG systems must solve.


The LLM answers only using the context given (topK chunks), basic RAG alone cannot provide accurate chunks.

---

➡️ **Next:** [`hybrid_search_RAG.ipynb`](./hybrid_search_RAG.ipynb) — add BM25 keyword search, fuse it with semantic search, and rerank the results.